# 03 — Ablaciones (8 variantes del set crítico)

Cada variante responde una amenaza metodológica del diseño E5 — qué pasa si quitas OOD,
familiarity, anti-pareidolia, si congelas VQ en Fase 3, si descongelas parcialmente el
backbone, si desactivas el atractor o si comparas contra un transfer-learning puro de ResNet18.

Las 8 variantes se entrenan con la misma metodología (3 fases) y se evalúan en clean + corrupt
(suite 4×3). Resultados consolidados en `out/ablation_summary.csv` / `.md`.

`RUN_TRAINING=True` entrena cada variante secuencialmente (~ horas en una GPU). `RUN_TRAINING=False`
carga checkpoints existentes desde `out/<variant>/best.pt`; si faltan, intenta sembrarlos desde
`experiments/atracctor/out/artifacts/dememte_e5_critical/seed_42/<variant>/`.

In [ ]:
import sys, os
from pathlib import Path
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'dememte').exists():
    ROOT = ROOT.parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
print('repo root:', ROOT)

In [ ]:
import json, shutil
from dataclasses import asdict

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from dememte.config import ABLATION_SPECS, ablation_config, BaselineConfig, resolve_data_dir
from dememte.data import build_loaders, seed_everything
from dememte.models import make_dememte_variant, ResNetBaseline
from dememte.training import train_dememte_full, train_baseline_phased
from dememte.evaluation import (
    evaluate_dememte_suite, evaluate_baseline_suite,
    signal_curve_rows, signal_curve_rows_baseline,
)
from dememte.io import save_checkpoint, load_checkpoint, write_json, write_csv, ensure_dir

RUN_TRAINING = False

device = 'cuda' if torch.cuda.is_available() else 'cpu'
OUT = ensure_dir(ROOT / 'notebooks' / '03_ablations' / 'out')
LEGACY_DIR = ROOT / 'experiments/atracctor/out/artifacts/dememte_e5_critical/seed_42'

data_dir = resolve_data_dir(BaselineConfig())
seed_everything(42)

## Datos (compartidos entre todas las variantes)

In [ ]:
tr_loader, va_loader, te_loader, meta = build_loaders(
    data_dir=data_dir,
    batch_size=16,
    num_workers=2,
    val_ratio=0.2,
    split_seed=42,
    protocol='historical_trainval_resplit',
)
print(meta)

## Variantes del set crítico

In [ ]:
VARIANTS = list(ABLATION_SPECS.keys())
for name in VARIANTS:
    print(f'  - {name:42s}  ::  {ABLATION_SPECS[name]["label"]}')

## Loop de entrenamiento / carga + evaluación

In [ ]:
all_summaries = []
all_curves = []

for variant in VARIANTS:
    print(f'\n=== {variant} ===')
    vdir = ensure_dir(OUT / variant)
    ckpt_path = vdir / 'best.pt'

    if variant == 'resnet18_transfer_baseline':
        cfg = BaselineConfig(freeze_backbone=True)
        cfg.data_dir = data_dir
        model = ResNetBaseline(num_classes=cfg.num_classes, freeze_backbone=True).to(device)
        if RUN_TRAINING:
            model, best_val = train_baseline_phased(model, tr_loader, va_loader, cfg, device)
            save_checkpoint(model, ckpt_path, extra={'best_val': best_val, 'variant': variant})
        else:
            legacy = LEGACY_DIR / variant
            legacy_pt = next(iter(legacy.glob('*best.pt')), None) if legacy.exists() else None
            if not ckpt_path.exists() and legacy_pt is not None:
                print('seeding from legacy:', legacy_pt)
                shutil.copy(legacy_pt, ckpt_path)
            if ckpt_path.exists():
                load_checkpoint(model, ckpt_path, device=device, strict=False)
            else:
                print('no checkpoint; skip variant'); continue
        metrics = evaluate_baseline_suite(model, te_loader, device=device)
        clean_record = metrics.pop('clean_record'); corrupt_records = metrics.pop('corruption_records')
        curve_rows = signal_curve_rows_baseline(variant, clean_record, corrupt_records)
    else:
        cfg = ablation_config(variant)
        cfg.data_dir = data_dir
        seed_everything(cfg.seed)
        model = make_dememte_variant(cfg, device=device)
        if RUN_TRAINING:
            model, best_val = train_dememte_full(model, tr_loader, va_loader, cfg, device)
            save_checkpoint(model, ckpt_path, extra={'best_val': best_val, 'variant': variant, 'config': asdict(cfg)})
        else:
            legacy = LEGACY_DIR / variant
            legacy_pt = next(iter(legacy.glob('*best.pt')), None) if legacy.exists() else None
            if not ckpt_path.exists() and legacy_pt is not None:
                print('seeding from legacy:', legacy_pt)
                shutil.copy(legacy_pt, ckpt_path)
            if ckpt_path.exists():
                load_checkpoint(model, ckpt_path, device=device, strict=False)
            else:
                print('no checkpoint; skip variant'); continue
        metrics = evaluate_dememte_suite(model, te_loader, device=device)
        clean_record = metrics.pop('clean_record'); corrupt_records = metrics.pop('corruption_records')
        curve_rows = signal_curve_rows(variant, ABLATION_SPECS[variant]['label'], clean_record, corrupt_records)

    summary = {k: v for k, v in metrics.items() if isinstance(v, (int, float, bool))}
    summary.update({'variant': variant, 'label': ABLATION_SPECS[variant]['label']})
    write_json(summary, vdir / 'metrics.json')
    all_summaries.append(summary)
    all_curves.extend(curve_rows)
    print(f"clean_acc={summary.get('clean_acc'):.4f} corrupt_acc={summary.get('corrupt_acc_avg'):.4f}")

summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(OUT / 'ablation_summary.csv', index=False)
write_csv(all_curves, OUT / 'ablation_curves.csv')
summary_df

## Tabla comparativa (clean vs corrupt)

In [ ]:
cols = ['variant', 'clean_acc', 'corrupt_acc_avg', 'corrupt_acc_gaussian_noise', 'corrupt_acc_pixel_mask', 'corrupt_acc_cutout', 'corrupt_acc_blur']
cols = [c for c in cols if c in summary_df.columns]
ranked = summary_df[cols].sort_values('corrupt_acc_avg', ascending=False)
ranked.to_markdown(OUT / 'ablation_summary.md', index=False) if hasattr(ranked, 'to_markdown') else ranked.to_csv(OUT / 'ablation_summary.md', sep='|', index=False)
ranked

## Comparativa visual clean vs corrupt

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(ranked))
width = 0.35
ax.bar(x - width/2, ranked['clean_acc'], width, label='clean')
ax.bar(x + width/2, ranked['corrupt_acc_avg'], width, label='corrupt avg')
ax.set_xticks(x); ax.set_xticklabels(ranked['variant'], rotation=35, ha='right')
ax.set_ylabel('Accuracy'); ax.set_title('Ablations — clean vs corrupt (suite 4×3)')
ax.grid(alpha=0.3, axis='y'); ax.legend()
ensure_dir(OUT / 'plots')
fig.savefig(OUT / 'plots' / 'ablations_clean_vs_corrupt.png', dpi=120, bbox_inches='tight')
plt.tight_layout(); plt.show()